# Notebook 13 – EDA Report


## Objective
Create a documented Exploratory Data Analysis (EDA) report. This notebook identifies data issues and recommends future preprocessing; 

### How to use
1. Put your CSV file in the notebook folder.
2. Set `DATA_PATH` in the setup cell.
3. Set `TARGET_COLUMN` and optionally update the domain/source details below.
4. Run cells from top to bottom.
5. Review generated findings before writing the final report.

## 1. Setup and load data
The fallback dataset is only for demonstration. Replace it with your actual dataset for a real report.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

# Option A: place your dataset CSV in the same folder and update this path.
DATA_PATH = "your_dataset.csv"

# Option B: if the file is not available, create a small example dataset so every cell can run.
if Path(DATA_PATH).exists():
    df = pd.read_csv(DATA_PATH)
    print(f"Loaded dataset: {DATA_PATH}")
else:
    rng = np.random.default_rng(42)
    n = 200
    df = pd.DataFrame({
        "customer_age": rng.integers(18, 70, n),
        "monthly_income": rng.normal(45000, 15000, n).round(0),
        "monthly_spend": rng.normal(18000, 7000, n).round(0),
        "visits_per_month": rng.poisson(4, n),
        "membership": rng.choice(["Basic", "Silver", "Gold"], n, p=[0.5, 0.3, 0.2]),
        "region": rng.choice(["North", "South", "East", "West"], n),
        "churn": rng.choice(["Yes", "No"], n, p=[0.25, 0.75])
    })
    # Add a few realistic data-quality issues for demonstration.
    df.loc[[3, 17, 41], "monthly_income"] = np.nan
    df.loc[[8, 55], "region"] = np.nan
    df.loc[12, "monthly_spend"] = 150000
    df = pd.concat([df, df.iloc[[0]]], ignore_index=True)
    print("Example dataset created because DATA_PATH was not found.")
print("Dataset shape:", df.shape)
display(df.head())

## 2. Dataset overview
Enter the dataset's source and domain accurately; the notebook cannot infer these reliably from the data alone.

In [ ]:
DATASET_SOURCE = "Update with file name, URL, or business source"
DOMAIN = "Update: e-commerce, banking, healthcare, retail, etc."
TARGET_COLUMN = "churn"  # Change to your target column, or use None if there is no target.

print("Source:", DATASET_SOURCE)
print("Domain:", DOMAIN)
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Features:", list(df.columns))
print("Target:", TARGET_COLUMN if TARGET_COLUMN in df.columns else "Not configured / not found")
display(pd.DataFrame({
    "column": df.columns,
    "dtype": [str(t) for t in df.dtypes],
    "non_null_count": [int(df[c].notna().sum()) for c in df.columns],
    "missing_count": [int(df[c].isna().sum()) for c in df.columns],
    "missing_percent": [round(df[c].isna().mean()*100, 2) for c in df.columns],
}))

## 3. Data quality assessment
These checks report potential issues only. Decide how to handle each issue during preprocessing.

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
print("Missing values by column:")
display(missing[missing > 0].to_frame("missing_count"))
print("Exact duplicate rows:", int(df.duplicated().sum()))

# A simple scan for suspicious negative values. Review context before deciding whether a value is invalid.
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
negative_counts = (df[numeric_cols] < 0).sum() if numeric_cols else pd.Series(dtype=int)
print("Negative values in numeric columns (review by domain):")
display(negative_counts[negative_counts > 0].to_frame("negative_count"))

print("Data types:")
display(df.dtypes.astype(str).rename("dtype").to_frame())
print("Potential high-cardinality categorical columns:")
cat_cols = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
high_cardinality = {c: int(df[c].nunique(dropna=True)) for c in cat_cols if df[c].nunique(dropna=True) > 30}
display(pd.Series(high_cardinality, name="unique_categories").to_frame())

## 4. Statistical findings
Summary statistics help describe central tendency, spread, and unusual ranges.

In [ ]:
display(df.describe(include="all").T)
if numeric_cols:
    stats = df[numeric_cols].agg(["mean", "median", "std", "min", "max"]).T
    stats["range"] = stats["max"] - stats["min"]
    stats["skewness"] = df[numeric_cols].skew(numeric_only=True)
    display(stats)

## 5. Outlier screening using IQR
IQR flags potential outliers; it does not prove that a record is wrong. Do not remove values automatically.

In [ ]:
outlier_rows = []
for col in numeric_cols:
    s = df[col].dropna()
    if s.empty:
        continue
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    count = int(((s < lower) | (s > upper)).sum())
    outlier_rows.append({"feature": col, "Q1": q1, "Q3": q3, "IQR": iqr,
                         "lower_bound": lower, "upper_bound": upper,
                         "potential_outlier_count": count})
outlier_summary = pd.DataFrame(outlier_rows)
display(outlier_summary)

## 6. Relationship findings
Correlation measures linear association between numeric variables. It does not establish causation.

In [ ]:
if len(numeric_cols) >= 2:
    corr = df[numeric_cols].corr(numeric_only=True)
    display(corr.round(2))
    plt.figure(figsize=(max(7, len(numeric_cols)), max(5, len(numeric_cols)*0.7)))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
    plt.title("Numeric Feature Correlation Heatmap")
    plt.tight_layout()
    plt.show()
else:
    print("At least two numeric columns are needed for a correlation heatmap.")

if TARGET_COLUMN in df.columns:
    print("Target distribution:")
    display(df[TARGET_COLUMN].value_counts(dropna=False).to_frame("count"))
    display((df[TARGET_COLUMN].value_counts(normalize=True, dropna=False)*100).round(2).to_frame("percent"))

## 7. Visualization findings
Use these plots to investigate distributions and relationships. Write down what each chart suggests.

In [ ]:
for col in numeric_cols[:6]:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    sns.histplot(df[col], kde=True, ax=axes[0])
    axes[0].set_title(f"Distribution: {col}")
    sns.boxplot(x=df[col], ax=axes[1])
    axes[1].set_title(f"Potential outliers: {col}")
    plt.tight_layout()
    plt.show()

for col in cat_cols[:5]:
    plt.figure(figsize=(7, 3.5))
    order = df[col].value_counts(dropna=False).index[:15]
    sns.countplot(data=df, x=col, order=order)
    plt.title(f"Category counts: {col}")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.show()

## 8. Potential data problems and next-sprint recommendations
Fill in this table based on the actual outputs. Recommendations are proposals, not actions performed in this notebook.

In [ ]:
recommendations = pd.DataFrame([
    {"potential_problem": "Missing values", "evidence_to_record": "Columns and missing percentages from Section 3",
     "possible_next_step": "Assess why values are missing; choose deletion or imputation based on domain and target."},
    {"potential_problem": "Duplicate records", "evidence_to_record": f"{int(df.duplicated().sum())} exact duplicate rows",
     "possible_next_step": "Confirm whether duplicates are genuine repeated events before removing."},
    {"potential_problem": "Potential outliers", "evidence_to_record": "IQR summary in Section 5",
     "possible_next_step": "Validate with domain knowledge; consider retaining, capping, transforming, or correcting only when justified."},
    {"potential_problem": "Categorical features", "evidence_to_record": f"{len(cat_cols)} categorical-like columns detected",
     "possible_next_step": "Choose one-hot, ordinal, or other encoding based on category meaning and model."},
    {"potential_problem": "Feature scale / skew", "evidence_to_record": "Review numeric summary and histograms",
     "possible_next_step": "Consider scaling or transformation if useful for the selected model."},
    {"potential_problem": "Data types", "evidence_to_record": "Review dtypes and date-like columns",
     "possible_next_step": "Correct types and parse dates in preprocessing after validation."},
])
display(recommendations)